<a href="https://colab.research.google.com/github/Malaya-Kumar-Pradhan/FlyRank-ML-01/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Malaya-Kumar-Pradhan/FlyRank-ML-01/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In the starter dataset (df), one row = one pseudonymized content item (content_id), representing aggregate, observed performance metrics over a trailing 90-day window.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")


Working dir: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter
Starter data found. You're ready.


In [ ]:
import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df.shape[0], "rows,", df.shape[1], "columns")
df.head(3)

30000 rows, 44 columns


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9


In [ ]:
# 1. Verify the grain: Is there exactly one row per content item?
is_unique = df['content_id'].is_unique
print(f"Is 'content_id' perfectly unique? {is_unique}")

# 2. Grain-probe: Ensure no content_id appears more than once
duplicates = df.groupby('content_id').size()
max_rows_per_id = duplicates.max()
print(f"Maximum rows per content_id: {max_rows_per_id} (Expected: 1)")

# 3. Confirm expected volume
print(f"Total rows (content items): {len(df):,}")

Is 'content_id' perfectly unique? True
Maximum rows per content_id: 1 (Expected: 1)
Total rows (content items): 30,000


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

#### **1. Feature (Inputs for modeling)**
Knowable before or at the time of evaluation; safe for model training.
* SEO & Target Metrics: search_volume, competition, competition_level, cpc, main_intent
* Content Properties: content_type, word_count, char_count, provider_used, model_used
* Measured Performance (Trailing 90-day & 30-day windows): ctr, avg_position, engagement_rate, scroll_rate, ai_traffic_pct, impressions_90d, clicks_90d, pageviews_90d, sessions_90d, users_90d, engaged_sessions_90d, ai_sessions_90d, scroll_events_90d, days_with_impressions, days_with_sessions, impressions_last_30d, clicks_last_30d, sessions_last_30d, content_age_days, days_since_last_update2.

#### **2. Label / Proxy (Targets)**
The target variables or metrics being predicted/evaluated. Never used as features.
* Primary Target: trend_direction (used for directional performance shift classification and scoring)

#### **3. Context (Grouping & Evaluation)**
Used for grouping, filtering, splitting, or human readability; excluded from model training features to avoid bias or overfitting.

* Identifiers: content_id (Primary Key / Unit of analysis), client_id (used strictly for group splitting/cross-validation to prevent data leakage)

#### **4. Excluded (Unusable / Harmful)**
Fields deliberately omitted from analysis or modeling, accompanied by a precise why.
* trend_pct: Excluded from direct model features to prevent target leakage, as it is the continuous raw calculation backing the target label (trend_direction).
* age_tier, age_tier_order, freshness_tier, word_count_tier, char_count_tier, impression_tier, position_tier: Excluded to prevent collinearity and redundancy because their raw continuous counterparts (content_age_days, word_count, char_count, impressions_90d, avg_position) are already measured and used directly as features.

* impressions_prev_30d, clicks_prev_30d, sessions_prev_30d: Excluded to avoid window-overlap data leakage, as historical lagging comparisons are captured cleanly via separate window differentials or model lagging logic.

In [22]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Define column buckets without overlap
features = [
    'search_volume', 'competition', 'competition_level', 'cpc', 'main_intent',
    'content_type', 'word_count', 'char_count', 'provider_used', 'model_used',
    'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d',
    'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d',
    'days_with_impressions', 'days_with_sessions', 'impressions_last_30d',
    'clicks_last_30d', 'sessions_last_30d', 'content_age_days',
    'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate',
    'scroll_rate', 'ai_traffic_pct'
]

label = ['trend_direction']

context = ['content_id', 'client_id']

# Automatically capture all remaining columns into excluded
assigned_cols = set(features + label + context)
excluded = [col for col in df.columns if col not in assigned_cols]

print(f"Total columns in dataset: {df.shape[1]}")
print(f"Features: {len(features)} | Label: {len(label)} | Context: {len(context)} | Excluded: {len(excluded)}")

# Strict assertion to guarantee complete coverage
assert len(features) + len(label) + len(context) + len(excluded) == df.shape[1], "Field count mismatch!"
print("Verification passed: All 44 columns successfully accounted for.")

Total columns in dataset: 44
Features: 30 | Label: 1 | Context: 2 | Excluded: 11
Verification passed: All 44 columns successfully accounted for.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [23]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# 3. Verify it with queries: Missing values and data distribution checks
# 1. Missingness per column (observed blank rates)
missing_df = pd.DataFrame({
    'missing_count': df.isnull().sum(),
    'missing_pct': (df.isnull().mean() * 100).round(2)
})
print("Columns with missing values:")
print(missing_df[missing_df['missing_count'] > 0])

# 2. Grouped missingness check by content_type to catch patterns
if 'content_type' in df.columns:
    print("\nMissingness percentage grouped by content_type (sample columns):")
    sample_cols = ['search_volume', 'cpc', 'ctr', 'avg_position']
    existing_sample = [c for c in sample_cols if c in df.columns]
    missing_by_type = df.groupby('content_type')[existing_sample].apply(lambda x: (x.isnull().mean() * 100).round(2))
    print(missing_by_type)

# 3. Summary stats to verify expected counts and value ranges
print(f"\nTotal verified rows: {len(df):,}")
print(f"Total columns managed: {df.shape[1]}")

Columns with missing values:
                   missing_count  missing_pct
search_volume               2468         8.23
competition                 2468         8.23
competition_level           2610         8.70
cpc                         2468         8.23
main_intent                 2374         7.91
word_count                  7699        25.66
char_count                  7699        25.66
provider_used              21438        71.46
model_used                  5733        19.11
word_count_tier             7699        25.66
char_count_tier             7699        25.66
scroll_rate                  125         0.42
trend_pct                   3388        11.29

Missingness percentage grouped by content_type (sample columns):
                    search_volume     cpc  ctr  avg_position
content_type                                                
comparison article           0.00    0.00  0.0           0.0
feedly article             100.00  100.00  0.0           0.0
keyword article  

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

* Unbalanced Historical Windows: The dataset provides measured performance metrics over static trailing windows (e.g., 90-day and 30-day aggregates) but cannot tell us how content performed prior to creation or prior to tracking initiation. Historical tracking is right-censored per item, meaning earlier baseline shifts are unmeasured.

* Granular Traffic Attribution (GSC-Only Early Rows): Because core impressions and clicks rely on aggregated search console feeds, the data cannot tell us exact intra-day user journeys, referrer paths, or session-level user intent transitions beyond what is captured via observed aggregate counts.

* Window Overlap Confounding: The simultaneous inclusion of 90-day and 30-day rolling metrics creates temporal overlaps. The dataset cannot cleanly isolate daily variance or independent weekly spikes without multi-window alignment, restricting trend conclusions to directional indicators rather than precise causal timelines.

* Operational Decision Context: Performance metrics serve purely as decision-support tools to guide content refresh prioritization; the data can never directly prescribe automated business actions or predict external market shifts outside the measured parameter space.

In [24]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# 4. Data limits check: Assessing limitations like unbalanced history or missing foundational tracking
print("Data limits analysis:")
print(f"Total rows: {len(df):,}")
print(f"Missing history check - Total null records across core traffic metrics:")
print(df[['impressions_90d', 'clicks_90d', 'sessions_90d']].isnull().sum())

# Check for rows with zero or incomplete active history windows
zero_impression_rows = (df['impressions_90d'] == 0).sum()
print(f"Rows with zero measured impressions in the 90-day window: {zero_impression_rows:,}")

Data limits analysis:
Total rows: 30,000
Missing history check - Total null records across core traffic metrics:
impressions_90d    0
clicks_90d         0
sessions_90d       0
dtype: int64
Rows with zero measured impressions in the 90-day window: 0


## Self-check

Before you submit, confirm each line honestly:

- [v] Every section above is filled — markdown thinking AND the code that backs it
- [v] The notebook runs top to bottom with no errors (Runtime → Run all)
- [v] No client names, URLs, or private queries anywhere
- [v] My claims use careful words: observed, measured, directional, decision-support
- [v] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.